# Pip 3D Model Generation via TripoSR

This notebook converts a 2D reference image of Pip the fox into a 3D model using **TripoSR**, an open-source image-to-3D model by Stability AI and Tripo.

**Runtime requirement:** This notebook MUST run on a GPU. Before starting:
1. Click **Runtime** → **Change runtime type**
2. Select **GPU** (e.g., T4 or V100)
3. Click **Save**

Then run all cells in order.

## Step 1: Install Dependencies

This cell installs TripoSR and all required packages. It may take 2–3 minutes.

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q git+https://github.com/VAST-AI-Research/TripoSR.git
!pip install -q pillow einops omegaconf
print("✓ Dependencies installed successfully")

## Step 2: Import Libraries and Set Up Device

Verify GPU is available and load the TripoSR model.

In [ ]:
import torch
from PIL import Image
import numpy as np
from pathlib import Path

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("GPU not available! Please change runtime to GPU in Runtime > Change runtime type")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"\n✓ Using device: {device}")

## Step 3: Load TripoSR Model

Load the pre-trained TripoSR model (may take 1–2 minutes on first run).

In [ ]:
from tripo_sr.models import TripoSRModel

print("Loading TripoSR model...")
model = TripoSRModel.from_pretrained_huggingface()
model.to(device)
model.eval()
print("✓ Model loaded successfully")

## Step 4: Upload Reference Image

Upload the 2D reference image. It should be a clean, well-lit image of Pip the fox against a simple background.

**Supported formats:** PNG, JPG, JPEG, WEBP

In [ ]:
from google.colab import files
import os

print("Upload your reference image (click 'Choose Files' below):")
uploaded = files.upload()

# Get the uploaded filename
image_path = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {image_path}")
print(f"  File size: {os.path.getsize(image_path) / 1e6:.2f} MB")

# Display the image
img = Image.open(image_path)
print(f"  Dimensions: {img.size}")
img.thumbnail((400, 400))
img.show()

## Step 5: Run TripoSR Inference

Generate the 3D mesh from the 2D image. This typically takes 1–3 minutes.

In [ ]:
from tripo_sr.utils import remove_background, resize_foreground
from PIL import Image

print("Preprocessing image...")
# Load image
image = Image.open(image_path).convert('RGB')

# Remove background (optional but recommended for better 3D results)
print("  Removing background...")
image_rembg = remove_background(image)

# Resize to model input size
print("  Resizing foreground...")
image_processed = resize_foreground(image_rembg, 0.85)

print("\nRunning TripoSR inference (this may take 1–3 minutes)...")
with torch.no_grad():
    mesh = model(image_processed, quality="medium")  # 'medium' for faster inference

print("✓ Inference complete!")
print(f"  Mesh vertices: {len(mesh.vertices)}")
print(f"  Mesh faces: {len(mesh.faces)}")

## Step 6: Export as GLB

Save the 3D mesh as a `.glb` file (binary GLTF format, suitable for Mixamo and other 3D tools).

In [ ]:
output_path = "pip_model.glb"
print(f"Exporting mesh to {output_path}...")
mesh.export(output_path)
file_size = os.path.getsize(output_path) / 1e6
print(f"✓ Exported successfully ({file_size:.2f} MB)")

## Step 7: Download the 3D Model

Download the generated `.glb` file to your machine. Save it to:

```
pip-pipeline/assets/meshes/pip_model.glb
```

In [ ]:
print(f"Downloading {output_path}...")
files.download(output_path)
print("✓ Download started in your browser")
print(f"\nMove the downloaded file to: assets/meshes/pip_model.glb")
print("Then proceed to Stage 2 (prep_for_mixamo.py)")

## Troubleshooting

**Issue: "CUDA out of memory" error**
- Try `quality="low"` in the model call (Step 5) for faster inference
- Or restart the runtime (Runtime > Restart runtime) and retry

**Issue: "No module named tripo_sr"**
- Re-run Step 1 (Install Dependencies) and wait for it to complete
- Restart the runtime and re-run all cells

**Issue: Background removal looks bad**
- Remove the `image_rembg = remove_background(image)` line and use the original image directly
- Or pre-process the image to remove the background before uploading